# 第 1 课作业：从膜电位到 LIF 的一步更新

这份作业对应课程 **第 1 课：从膜电位到一个最小计算神经元**。

本题练的不是 Python 技巧，而是把漏积分发放模型（Leaky Integrate-and-Fire, LIF）的一步更新写成明确、可检查的规则。完成以后，你应该能把 **leak → input → threshold → reset** 四个动作分别指出来。

## 本题目标与规则

本课固定使用：

$$
V_{candidate}=V_{rest}+\alpha(V-V_{rest})+I
$$

并规定：

- 先计算 candidate voltage；
- candidate 达到或超过 threshold 时产生 spike；
- spike 后立即把下一步保存的状态设为 reset；
- 没有 spike 时保存 candidate；
- 不修改函数签名。

## 先不用代码：手算一步

先在纸上计算，不要运行 Python。

给定：

- 当前膜电位 V = -62
- V_rest = -70
- alpha = 0.5
- input = 3

请先算出 candidate voltage。

然后分别考虑两种 threshold：

1. threshold = -60
2. threshold 恰好等于你刚算出的 candidate

对每种情况写出：

- 是否 spike；
- 下一步真正保存的电位是多少（假设 reset = -70）。

这一步的目的，是先把 **candidate** 和 **stored state** 分开，再写代码。

## Part A：实现 leak_step()

### 这个函数做什么？

`leak_step()` 只计算 **leak + input integration** 之后得到的候选膜电位（candidate voltage）。

它**不判断 threshold，也不产生 spike，更不做 reset**。可以把它理解成：“如果暂时不考虑是否放电，这一步膜电位会走到哪里？”

### 输入

- `v`：当前时间步开始时的膜电位；
- `input_value`：这个时间步加入的输入电流；
- `alpha`：上一时刻膜电位偏离静息电位后，还保留多少的系数；
- `v_rest`：静息膜电位。

### 输出

函数只返回 **一个浮点数**：

- candidate voltage：完成 leak 和 input integration 后、但还没有做 threshold/reset 判断的膜电位。

也就是说，这个函数的输出仍然只是“候选值”，不一定就是下一时间步真正保存的膜电位。

In [ ]:
def leak_step(v: float, input_value: float, alpha: float, v_rest: float) -> float:
    # YOUR CODE STARTS HERE
    raise NotImplementedError("TODO: implement one leak + integrate step")
    # YOUR CODE ENDS HERE

## Part B：实现 lif_step()

### 这个函数做什么？

`lif_step()` 完成 **一个完整的 LIF 时间步更新**。

它先复用 Part A 的 `leak_step()` 得到 candidate voltage，然后：

1. 判断 candidate 是否达到或超过 threshold；
2. 如果达到 threshold，本次更新产生 spike，并把下一时间步保存的膜电位设为 `reset`；
3. 如果没有达到 threshold，本次更新不产生 spike，下一时间步保存 candidate 本身。

### 输入

前四个参数与 Part A 相同：

- `v`：当前时间步开始时的膜电位；
- `input_value`：本时间步输入；
- `alpha`：leak 系数；
- `v_rest`：静息膜电位。

另外还有：

- `threshold`：spike threshold；
- `reset`：发生 spike 后，写入下一时间步的膜电位。

### 输出

这个函数返回 **两个值**，顺序固定为：

`(next_v, spike)`

其中：

1. `next_v`：一个浮点数，表示**本次更新结束后、要保存给下一时间步使用的膜电位**。  
   - 如果产生 spike，`next_v = reset`；
   - 如果没有 spike，`next_v = candidate_v`。

2. `spike`：一个布尔值（Boolean）。  
   - `True`：表示**本次时间步更新产生了 spike**；
   - `False`：表示本次更新没有产生 spike。

注意：`spike` 描述的是“这一次更新是否发生放电事件”，而 `next_v` 描述的是“更新之后留下什么状态给下一时间步”。这两个信息要同时返回。

In [ ]:
def lif_step(
    v: float,
    input_value: float,
    alpha: float,
    v_rest: float,
    threshold: float,
    reset: float,
) -> tuple[float, bool]:
    candidate_v = leak_step(v, input_value, alpha, v_rest)

    # YOUR CODE STARTS HERE
    raise NotImplementedError("TODO: implement threshold and reset")
    # YOUR CODE ENDS HERE

    return next_v, spike

## 检查你的实现

先确认 Part A、Part B 两个代码单元都已经运行。下面只调用外部 grader；具体测试输入不会显示在作业 Notebook 中。

In [ ]:
# Course infrastructure: make the repository root importable from a notebook subdirectory.
from pathlib import Path
import sys

_repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "exercises" / "grader").is_dir()
)
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from exercises.grader.lesson01 import check

check(leak_step=leak_step, lif_step=lif_step, language="zh")

## Human Check

不用运行代码，指着你自己的实现回答：

1. leak_step() 的哪一部分保证 V = V_rest 且输入为 0 时，V_rest 是固定点？
2. lif_step() 中哪一步决定“等于 threshold 也会 spike”？
3. spike 的那个时间步里，candidate 和真正保存到下一步的 state 为什么可能不同？
4. 如果把 threshold/reset 逻辑偷偷塞进 leak_step()，为什么会让两个函数的职责变得不清楚？